# Embedding

Run `pipeline.embedder.embed_document` for each PDF that extracts text.
The last cell deletes this notebook's collection.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
os.chdir(ROOT)
for entry in (ROOT / "notebooks", ROOT):
    text = str(entry)
    if text not in sys.path:
        sys.path.insert(0, text)

from helpers import accepted_pdfs, new_session_id, timed

from pipeline.extractor import extract

pdf_dir = (
    Path(os.environ["NOTEBOOK_PDF_DIR"])
    if os.environ.get("NOTEBOOK_PDF_DIR")
    else ROOT / "notebooks" / "inputs"
)
session_id = new_session_id()
documents = []

with timed("setup") as setup:
    accepted, rejected = accepted_pdfs(pdf_dir)
    for path in accepted:
        try:
            document = await extract(path)
        except Exception as exc:
            print(path.name, "failure", type(exc).__name__, exc)
            continue
        if document.metadata.total_chunks == 0:
            print(path.name, "failure", "RuntimeError", "produced no text")
            continue
        documents.append(document)

print(session_id)
print(f"setup: {setup['seconds']:.3f}s")
print(f"accepted: {len(accepted)}  ready: {len(documents)}  rejected: {len(rejected)}")

In [ ]:
from pipeline.embedder import embed_document

embedded = []
for document in documents:
    with timed("embed") as timing:
        await embed_document(document, session_id)
    embedded.append((document, timing["seconds"]))

In [ ]:
from pipeline.vector_store import list_sources

sources = list_sources(session_id)
print("filename\tchunks\tsources\tseconds")
for document, seconds in embedded:
    print(
        document.metadata.filename,
        document.metadata.total_chunks,
        ", ".join(sources),
        f"{seconds:.3f}",
        sep="\t",
    )

In [ ]:
from pipeline.embedder import delete_collection

delete_collection(session_id)
print(session_id, "deleted")